# 02 — LoRA Walkthrough

Low-Rank Adaptation (LoRA) lets us fine-tune large models cheaply by adding
small trainable matrices to selected layers — while keeping the original
weights frozen.

This notebook covers:
1. Why LoRA works — the low-rank intuition
2. What rank, alpha, and dropout do
3. Which modules to target and why (`q_proj`, `v_proj`)
4. Why these settings are practical on Apple Silicon M4

## 1. The core idea

A transformer weight matrix `W` has shape `[d_out, d_in]`. Full fine-tuning
updates every element — for a 3B model, that's billions of trainable parameters.

LoRA's insight: the *task-specific update* `ΔW` needed to adapt a pre-trained
model to a narrow task tends to live in a low-dimensional subspace.

Instead of learning `ΔW` directly, LoRA decomposes it:

```
ΔW = B × A
```

where:
- `A` has shape `[rank, d_in]`  — projects input down to rank dimensions
- `B` has shape `[d_out, rank]` — projects back up to output dimensions
- `rank << min(d_out, d_in)`   — the key constraint that keeps it cheap

Only `A` and `B` are trained. The original `W` is frozen.

In [ ]:
# Concrete numbers: parameter count comparison for one attention layer
# in a Qwen2.5-3B model

d_model = 2048  # hidden dim for Qwen2.5-3B
d_head = 128
n_heads = 16
d_kv = d_head * n_heads  # 2048

# Full weight matrices (frozen in LoRA)
W_q_params = d_model * d_model
W_v_params = d_model * d_model

# LoRA adapter parameters (trainable)
rank = 8
lora_q_params = d_model * rank + rank * d_model  # A + B
lora_v_params = d_model * rank + rank * d_model

total_lora = lora_q_params + lora_v_params
total_full = W_q_params + W_v_params

print(f"Full fine-tune (q_proj + v_proj):   {total_full:>12,} params")
print(f"LoRA rank={rank} (q_proj + v_proj): {total_lora:>12,} params")
print(f"LoRA reduction factor:              {total_full/total_lora:.0f}x fewer")

## 2. The key hyperparameters

### rank

Controls the expressiveness of the adapter — how many dimensions the update
is allowed to occupy.

- **rank=8** is the standard starting point for instruction fine-tuning on 3B models.
- Lower rank (4) is faster and uses less memory; riskier for structured output tasks.
- Higher rank (16, 32) allows richer adaptation if quality plateaus.
- Rule of thumb: start low, increase only when eval shows the model can't learn the task.

### alpha

Scales the LoRA update: the effective update is `(alpha / rank) * ΔW`.

- Setting `alpha = rank` gives a unit-scale update.
- Setting `alpha = 2 * rank` (our default: alpha=16, rank=8) amplifies the adapter
  signal slightly — often helps when fine-tuning from a strong base model.
- If training is unstable, reduce alpha toward rank.

### dropout

Applied to the LoRA matrices during training to prevent overfitting on small datasets.
0.05 is conservative. Increase to 0.1 if you see clear overfitting on val loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize how rank affects the number of trainable parameters
d = 2048  # hidden dim
ranks = [4, 8, 16, 32, 64]
# q_proj + v_proj LoRA params
params = [2 * (d * r + r * d) for r in ranks]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([str(r) for r in ranks], [p/1e6 for p in params], color="#4A90D9")
ax.bar_label(bars, fmt="%.1fM", padding=3)
ax.set_xlabel("LoRA rank")
ax.set_ylabel("Trainable parameters (M)")
ax.set_title("LoRA adapter size vs rank\n(q_proj + v_proj, d=2048)")
ax.axvline(x=1, color="red", linestyle="--", alpha=0.5, label="default (rank=8)")
ax.legend()
plt.tight_layout()
plt.show()

print("Memory for full q+v weights (frozen): ", f"{2 * d * d * 4 / 1e6:.0f} MB (fp32)")
print("Memory for rank=8 adapters (trainable):", f"{params[1] * 4 / 1e6:.1f} MB (fp32)")

## 3. Why q_proj and v_proj?

Transformer attention has four projection matrices: Q, K, V, O.

- **Q (query)** — determines what each token is looking for
- **K (key)** — determines what each token presents to others
- **V (value)** — the actual information retrieved
- **O (output)** — projects the attended values back to the residual stream

**Q and V are the highest-leverage targets** for task-specific fine-tuning:
- Q shapes *attention patterns* — what the model focuses on given a prompt
- V shapes *what gets retrieved* — the content extracted from the context

For structured extraction tasks (like Trace Layer 2), Q+V adaptation is usually
sufficient to teach the model new output schemas and tier classification behavior.

**Adding K and O** (see config `# - k_proj / o_proj`):
- K shapes what each token exposes to attention queries — useful for richer relational
  understanding between trace segments
- O shapes the write-back into the residual stream — useful for nuanced reasoning
- Cost: ~2x adapter parameters; well within M4 budget
- Recommended for Stage 2+ when Q+V quality has plateaued

In [ ]:
# Adapter parameter budget across different target module configurations
d = 2048
rank = 8

configs = {
    "q + v (default)": 2,
    "q + k + v": 3,
    "q + k + v + o": 4,
}

print("Adapter parameter budget by target module config (rank=8, d=2048):")
print()
for label, n_mats in configs.items():
    params = n_mats * 2 * d * rank  # A + B for each matrix
    print(f"  {label:<25} {params:>9,} params  ({params*4/1e6:.2f} MB fp32)")

## 4. Apple Silicon M4 considerations

MLX is designed for Apple Silicon's unified memory architecture. Key points:

**Why MLX is practical here:**
- Unified memory means CPU and GPU share the same pool (64GB on your M4 Pro)
- No PCIe data transfer overhead — weights stay in unified memory during training
- 4-bit quantized models (e.g. Qwen2.5-3B-4bit ≈ 2GB) leave ample headroom
  for activations, optimizer state, and the LoRA adapter

**Typical memory budget for SFT on Qwen2.5-3B-4bit:**
- Model weights (frozen, 4-bit): ~2 GB
- LoRA adapters (fp32, rank=8, q+v): ~0.03 GB
- Activations per batch (batch=4, seq=512): ~1–2 GB
- Total: ~4–6 GB — easily within 64GB

**Why `grad_checkpoint: true`:**
- Gradient checkpointing recomputes activations on the backward pass instead of
  storing them all in memory during the forward pass
- Trades ~30% more compute for significantly lower memory per batch
- Recommended even on 64GB to keep headroom for larger batch sizes later

In [ ]:
# Rough memory budget estimates
model_4bit_gb = 2.0       # Qwen2.5-3B at 4-bit quantization
lora_adapter_mb = 30      # rank=8, q+v, fp32 (from calculation above)
activation_per_sample_mb = 200  # rough estimate at seq_len=512
batch_sizes = [1, 2, 4, 8]

print("Estimated peak memory usage during SFT:")
print(f"  Base model (frozen, 4-bit):  {model_4bit_gb*1024:.0f} MB")
print(f"  LoRA adapters (fp32):         {lora_adapter_mb} MB")
print()
print(f"  {'Batch size':<15} {'Activation GB':<18} {'Total est. GB':<15}")
print("  " + "-" * 48)
for bs in batch_sizes:
    act_gb = bs * activation_per_sample_mb / 1024
    total_gb = model_4bit_gb + lora_adapter_mb/1024 + act_gb
    headroom = 64 - total_gb
    print(f"  {bs:<15} {act_gb:<18.2f} {total_gb:<12.1f} ({headroom:.0f} GB free of 64)")